In [ ]:
# ===============================================================================
# CELL 1: INSTALLATION, FUNCTIONS, AND SINGLE EXECUTION EXAMPLE
# ===============================================================================
!pip install --quiet google-api-python-client pandas

import os
import time
import pandas as pd
from google.colab import drive
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import nltk
from nltk.corpus import stopwords

In [ ]:
# --- NLTK Downloads (run only once) ---
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('taggers/averaged_perceptron_tagger')
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('averaged_perceptron_tagger', quiet=True)
    nltk.download('stopwords', quiet=True)

In [ ]:
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# --- Mount Google Drive ---
try:
    drive.mount('/content/drive')
except:
    print("Drive already mounted or error mounting.")


In [ ]:
# ==============================================================================
# EXTRACTION FUNCTIONS AND API QUERY
# ==============================================================================

def extract_key_terms(text):
    """Extracts nouns, verbs, and adjectives from a text, removing stopwords."""
    if not isinstance(text, str):
        return []
    tokens = nltk.word_tokenize(text.lower())
    english_stopwords = set(stopwords.words('english'))
    tagged_words = nltk.pos_tag(tokens)
    # Using a simple set of tags for robustness
    key_terms = [word for word, tag in tagged_words if tag.startswith(('NN', 'VB', 'JJ')) and word not in english_stopwords]
    return list(set(key_terms))

def get_google_kg_definitions(query_terms, service):
    """Searches for definitions for a list of terms in the Google Knowledge Graph."""
    if not query_terms:
        return 'No key concepts extracted.'
    definitions = []
    for term in query_terms:
        try:
            request = service.entities().search(query=term, languages='en', limit=1)
            response = request.execute()
            if 'itemListElement' in response and response['itemListElement']:
                result = response['itemListElement'][0]['result']
                name = result.get('name', 'N/A')
                final_description = "No description available."
                if 'detailedDescription' in result:
                    final_description = result['detailedDescription'].get('articleBody', "No detailed description text.")
                elif 'description' in result:
                    final_description = result.get('description')
                definitions.append(f"Query: '{term}' -> Result: '{name}'. Definition: {final_description}")
        except Exception:
            # Ignore errors for a single term to not stop the process
            continue
    if definitions:
        return ' '.join(list(set(definitions)))
    else:
        return 'No relevant definitions found for the key concepts.'

# ==============================================================================
# SINGLE CASE TEST WITH YOUR DATASET
# ==============================================================================

# --- Configuration ---
API_KEY = "" # Your API key here
DATASET_PATH = '' # Your dataset path here

print("--- Running test with the first row of the AbstRT.csv dataset ---")

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: Google API key not configured.")
else:
    try:
        # Load your dataset
        df_test = pd.read_csv(DATASET_PATH)

        # Get the first row for the example
        example_row = df_test.iloc[0]

        # Extract data from the row
        argument_text = example_row['argument']
        topic_text = example_row['topic']

        # Build the "question" in the expected format for the functions
        question_text = f'The argument "{argument_text}" ATTACK or SUPPORT the belief "{topic_text}"?'

        # Extract key concepts from the argument and topic
        argument_entities = extract_key_terms(argument_text)
        topic_entities = extract_key_terms(topic_text)
        all_entities = list(set(argument_entities + topic_entities))

        print(f"\nArgument: {argument_text}")
        print(f"Topic: {topic_text}")
        print(f"Extracted key concepts: {all_entities}")

        # Search for context in the Knowledge Graph
        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)
        start_time = time.time()
        knowledge_context = get_google_kg_definitions(all_entities, kg_service)
        end_time = time.time()

        print(f"\n⏱️ Query execution time: {end_time - start_time:.4f} seconds")
        print("\n--- Generated Prompt for the Example ---")

        # Build the final prompt with the UNDEFINED option
        final_prompt_example = (
            f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
            f"which shows direct paths between concepts from the argument and the belief. "
            f"Based on this, decide if the argument attacks or supports the belief. "
            f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
            f"Knowledge Graph Context: {knowledge_context}\n\n"
            f"Question: {question_text}"
        )

        print(final_prompt_example)

    except FileNotFoundError:
        print(f"❌ ERROR: File not found at: {DATASET_PATH}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
# ==============================================================================
# CELL 2: MAIN PROCESSING SCRIPT
# ==============================================================================
from tqdm.auto import tqdm

# --- Configuration ---
API_KEY = "" # Your API key here
INPUT_CSV_PATH = '' # Your dataset path here
OUTPUT_CSV_PATH = '' # Your output path here

if not API_KEY or API_KEY == "YOUR_API_KEY_HERE":
    print("⚠️ WARNING: Google API key not configured. The script cannot continue.")
else:
    # --- 1. DATA LOADING AND PREPROCESSING ---
    df = None
    try:
        df = pd.read_csv(INPUT_CSV_PATH)
        print(f"File '{INPUT_CSV_PATH}' loaded successfully. Total of {len(df)} rows.")

        # Create the 'question' column in the required format
        df['question'] = (
            "The argument \"" + df['argument'].astype(str) + "\" "
            "SUPPORT or ATTACK the belief \"" + df['topic'].astype(str) + "\"?"
        )
        print("Data preprocessing completed.")

    except FileNotFoundError:
        print(f"❌ ERROR: File not found at the specified path: {INPUT_CSV_PATH}")
        df = None
    except Exception as e:
        print(f"An unexpected error occurred during loading/preprocessing: {e}")
        df = None

    if df is not None:
        # --- 2. BATCH PROCESSING ---
        results_data = []
        kg_service = build('kgsearch', 'v1', developerKey=API_KEY, cache_discovery=False)

        print(f"\nStarting prompt generation for the {len(df)} samples in the dataset.")
        for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating prompts"):
            question_prompt = row['question']

            # Extract key concepts and search for context
            argument_entities = extract_key_terms(row['argument'])
            topic_entities = extract_key_terms(row['topic'])
            all_entities = list(set(argument_entities + topic_entities))
            knowledge_context = get_google_kg_definitions(all_entities, kg_service)

            # Build the final prompt with the UNDEFINED option
            final_prompt = (
                f"Your task is to analyze an argument and a belief, using the provided 'Knowledge Graph Context' "
                f"which shows direct paths between concepts from the argument and the belief. "
                f"Based on this, decide if the argument attacks or supports the belief. "
                f"Your answer must be ONLY the word 'ATTACK' or 'SUPPORT'.\n\n"
                f"Knowledge Graph Context: {knowledge_context}\n\n"
                f"Question: {question_prompt}"
            )

            # Add results to the list using the correct column names
            results_data.append({
                'subject': row['subject'],
                'argument': row['argument'],
                'topic': row['topic'],
                'knowledge_context': knowledge_context,
                'final_prompt': final_prompt,
                'ground_truth_stance': row['ground_truth_stance']
            })
            time.sleep(0.1) # Pause to avoid overloading the API

        # --- 3. SAVING THE RESULTS ---
        results_df = pd.DataFrame(results_data)
        results_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
        print(f"\n✅ Process completed! The file '{OUTPUT_CSV_PATH}' was saved successfully.")
        display(results_df.head())

In [ ]:
results_df.ground_truth_stance.value_counts()

In [ ]:
print(results_df.final_prompt[0])